# 00 · Quickstart: a muon and an electron in SBND

lumen follows optical photons through a detector and tells you how much light each light sensor sees, and when.
SBND, a liquid-argon detector, has 120 photomultiplier tubes (PMTs) and 192 X-ARAPUCA light collectors behind its
wire planes. You need no knowledge of derivatives for this notebook: we simulate two events and look at them.

What we use:

- **Geant4 steps**: Geant4 (the standard particle-physics simulation) tracked an 800 MeV muon and a 30 MeV electron
  through SBND's argon and wrote down every short piece of each charged track (a *step*: where it starts and ends,
  how fast the particle goes, the energy it leaves). lumen turns them into light: **scintillation** (argon glows
  when a charged particle passes, about 24,000 photons per MeV at 128 nm, in the vacuum ultraviolet) and
  **Cherenkov light** (the cone of light of a particle faster than light in argon).
- **A detector file**: SBND's geometry and optics, converted once from its GDML (the geometry file Geant4 reads).
  Notebook 01 shows the conversion; here it just happens if the file is missing.

Units everywhere: metres, nanoseconds, nanometres, electronvolts.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import lumen
import lumen_tpc
from lumen.compare import event, references
from lumen.tools import plot

WORK = Path(os.environ.get("LUMEN_NOTEBOOKS", "lumen-notebooks"))  # where the converted detector goes
WORK.mkdir(parents=True, exist_ok=True)

path = WORK / "sbnd.lumen"
if not path.exists():
    lumen.convert("lumen_tpc:sbnd", out=path, data=references.store())  # once, about a minute and a half
det = lumen.load(path)
det

## The two Geant4 events

Each event file holds the steps of one event (and the hits Geant4 recorded, which notebook 02 compares with).
`references.product` finds the files and checks they are the ones this package expects.

In [ ]:
events = references.product("g4ref-events-tpc-933e631d", extra=[lumen_tpc.REFERENCES])
muon = event.read(events / "sbnd_mu800.npz")
electron = event.read(events / "sbnd_e30.npz")
for name, ev in (("muon", muon), ("electron", electron)):
    print(f"{name}: {ev.meta['particle']} of {ev.meta['kinetic_energy_mev']} MeV from {ev.meta['position_mm']} mm "
          f"along {ev.meta['direction']}: {len(ev.steps['edep'])} steps, "
          f"{ev.steps['edep'].sum() / 1e6:.0f} MeV deposited")

## Simulate

`event.simulate` makes the light sources from the steps (scintillation and Cherenkov light of each material) and
runs `lumen.simulate` on each. `samples` is how many Monte Carlo photons lumen follows per source to estimate the
light; the physics (how many photons the particle really makes) comes from the steps. More samples only make the
estimate less noisy. The first call compiles the photon loop for this detector (about a minute); later calls take a
fraction of a second.

In [ ]:
light_mu = event.simulate(det, muon, samples=2**18, seed=0)
light_e = event.simulate(det, electron, samples=2**18, seed=0)
print("photons reaching the sensors: muon", round(float(light_mu.arrivals.sum())), " electron",
      round(float(light_e.arrivals.sum())))

Which light is it? Simulate each kind alone (`lights=`): scintillation dominates.

In [ ]:
for kind in ("scintillation", "cherenkov"):
    alone = event.simulate(det, muon, lights=(kind,), samples=2**18, seed=0)
    print(f"muon, {kind} only: {float(alone.arrivals.sum()):.0f} photons at the sensors")

## Charge and time per sensor

- `light.arrivals[i]`: expected photons reaching sensor *i*;
- `lumen.readout.pe(light, params)`: expected **photoelectrons** (the sensor's charge) after its detection efficiency,
  from each sensor type's values (coated and uncoated PMTs, X-ARAPUCAs for ultraviolet and for visible light);
- `light.mean_arrival_time[i]`: the average arrival time in ns. Argon's scintillation has a fast part (a few ns)
  and a slow part (about 1.5 µs), so the mean is hundreds of ns even though the first light comes within ns.

In [ ]:
pe_mu = lumen.readout.pe(light_mu, det.params())
pe_e = lumen.readout.pe(light_e, det.params())
types = np.array(det.sensor_table().columns["type"])  # each sensor's type, in the order of the results
for t in np.unique(types):
    m = types == t
    mu, e = float(pe_mu[m].sum()), float(pe_e[m].sum())
    print(f"{t:14s} {m.sum():4d} sensors: muon {mu:9.0f} PE, electron {e:7.0f} PE")

## Event displays

Each dot is a sensor. SBND's sensors sit on two planes, one behind each wire plane, facing each other across the
argon; the display shows them side by side (left to right: the angle around the detector's long axis; bottom to
top: along that axis). The muon starts in front of one plane and crosses the detector along its length.

In [ ]:
plot.event(det, pe_mu, light_mu.mean_arrival_time, title="muon")
plot.event(det, pe_e, light_e.mean_arrival_time, title="electron")
plt.show()

## How fast?

Now that the loop is compiled, time one event at two photon counts. The noise of each sensor's estimate falls as
1/sqrt(samples): four times the samples, half the noise. (Each error is estimated from the photons' own spread, each
sensor's per-photon second moment: on a sensor with tens of photons or more it is known to about 15 %.)

In [ ]:
import time

for n in (2**16, 2**18):
    event.simulate(det, muon, samples=n, seed=1)  # a new photon count compiles once: time the second call
    t0 = time.perf_counter()
    light = event.simulate(det, muon, samples=n, seed=2)
    light.arrivals.block_until_ready()
    i = int(np.argmax(np.asarray(light.arrivals)))
    rel = float(light.arrivals_error[i] / light.arrivals[i])
    print(f"{n:7d} samples: {time.perf_counter() - t0:.2f} s; brightest sensor's relative error {rel:.3f}")

**Next:** 01 converts the detector yourself; 02 checks lumen against Geant4; 03 explains what a derivative of a
simulation is; 04 uses derivatives to calibrate the argon's optics.